In [5]:
import os
import time
import duckdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Di Jupyter Notebook, __file__ tidak ada. Kita gunakan current working directory.
BASE_DIR = os.getcwd()

# Temp dir untuk DuckDB spill-to-disk saat sorting
DUCKDB_TEMP = os.path.join(BASE_DIR, ".duckdb_tmp")
os.makedirs(DUCKDB_TEMP, exist_ok=True)
DROPPED_DIR = os.path.join(BASE_DIR, "convert-to-parquet")
os.makedirs(DROPPED_DIR, exist_ok=True)

def duckdb_low_mem():
    """Buat koneksi DuckDB dengan limit RAM 2 GB, spill ke disk."""
    con = duckdb.connect()
    con.execute(f"SET temp_directory='{DUCKDB_TEMP}'")
    con.execute("SET memory_limit='2GB'")
    return con

# Variabel global untuk tracking ukuran file
total_csv_bytes = 0
total_pq_bytes = 0

In [6]:
# ==========================================
# 1. KONVERSI DataGPS (per bulan, gabung parts)
# ==========================================
GPS_SRC = os.path.join(BASE_DIR, "DataGPS")
GPS_DST = os.path.join(BASE_DIR, "DataGPS_parquet")
os.makedirs(GPS_DST, exist_ok=True)

# Mapping: nama output -> list file CSV sumber
gps_files = {
    "2021_10_Oktober": ["2021Oktober/Oktober2021.csv"],
    "2021_11_November": [f"2021November/November2021_part{i}.csv" for i in range(1, 8)],
    "2021_12_Desember": [f"2021Desember/Desember2021_part{i}.csv" for i in range(1, 4)],
    "2022_01_Januari": [f"2022Januari/Januari2022_part{i}.csv" for i in range(1, 3)],
    "2022_02_Februari": [f"2022Februari/Februari2022_part{i}.csv" for i in range(1, 3)],
    "2022_03_Maret": [f"2022Maret/Maret2022_part{i}.csv" for i in range(1, 3)],
    "2022_04_April": ["2022April/April2022.csv"],
    "2022_05_Mei": [f"2022Mei/Mei2022_part{i}.csv" for i in range(1, 3)],
    "2022_06_Juni": ["2022Juni/Juni2022.csv"],
}

GPS_SCHEMA = pa.schema([
    ("maid", pa.string()),
    ("latitude", pa.float64()),
    ("longitude", pa.float64()),
    ("timestamp", pa.int64()),
])

CHUNK_SIZE = 500_000

print("=" * 60)
print("KONVERSI CSV -> PARQUET")
print("=" * 60)

for month_name, csv_list in gps_files.items():
    t0 = time.time()
    out_path = os.path.join(GPS_DST, f"{month_name}.parquet")

    if os.path.exists(out_path):
        print(f" [SKIP] {month_name}.parquet sudah ada")
        total_pq_bytes += os.path.getsize(out_path)
        for cf in csv_list:
            fp = os.path.join(GPS_SRC, cf)
            if os.path.exists(fp):
                total_csv_bytes += os.path.getsize(fp)
        continue

    print(f"\n [{month_name}] Mengkonversi {len(csv_list)} file CSV...", flush=True)

    writer = None
    total_rows = 0
    month_csv_bytes = 0
    rows_dropped = 0

    for csv_file in csv_list:
        fpath = os.path.join(GPS_SRC, csv_file)
        if not os.path.exists(fpath):
            print(f" [WARN] File tidak ditemukan: {csv_file}")
            continue

        month_csv_bytes += os.path.getsize(fpath)

        for chunk in pd.read_csv(
            fpath,
            chunksize=CHUNK_SIZE,
            dtype={"maid": str, "latitude": str, "longitude": str, "timestamp": str},
        ):
            chunk_raw = chunk.copy()

            chunk["latitude"] = pd.to_numeric(chunk["latitude"], errors="coerce")
            chunk["longitude"] = pd.to_numeric(chunk["longitude"], errors="coerce")
            chunk["timestamp"] = pd.to_numeric(chunk["timestamp"], errors="coerce")

            before_drop = len(chunk)
            mask_nan = chunk[["maid", "latitude", "longitude", "timestamp"]].isna().any(axis=1)
            dropped_chunk = chunk_raw[mask_nan]
            chunk = chunk[~mask_nan]
            rows_dropped += before_drop - len(chunk)

            if not dropped_chunk.empty:
                # Mirror path: convert-to-parquet/DataGPS/2021Oktober/Oktober2021_dropped.csv
                rel_path = os.path.relpath(fpath, BASE_DIR)          # e.g. DataGPS/2021Oktober/Oktober2021.csv
                stem, ext = os.path.splitext(rel_path)
                dropped_csv_path = os.path.join(DROPPED_DIR, stem + "_dropped" + ext)
                os.makedirs(os.path.dirname(dropped_csv_path), exist_ok=True)
                write_header = not os.path.exists(dropped_csv_path)
                dropped_chunk.to_csv(dropped_csv_path, mode="a", index=False, header=write_header)

            chunk["latitude"] = chunk["latitude"].astype("float64")
            chunk["longitude"] = chunk["longitude"].astype("float64")
            chunk["timestamp"] = chunk["timestamp"].astype("int64")
            chunk["maid"] = chunk["maid"].astype(str)

            chunk = chunk[["maid", "latitude", "longitude", "timestamp"]]

            table = pa.Table.from_pandas(chunk, schema=GPS_SCHEMA, preserve_index=False)

            if writer is None:
                writer = pq.ParquetWriter(out_path, GPS_SCHEMA, compression="zstd")

            writer.write_table(table)
            total_rows += len(chunk)

    if writer is not None:
        writer.close()

    if rows_dropped > 0:
        print(f" [{month_name}] Dropped {rows_dropped:,} baris dengan NaN (latitude/longitude/timestamp)")

    # Sorting berdasarkan maid lalu timestamp (DuckDB out-of-core, hemat RAM)
    if os.path.exists(out_path):
        print(" Sorting berdasarkan maid, timestamp...", flush=True)
        t_sort = time.time()
        tmp_sorted = out_path + ".sorting.tmp"
        con = duckdb_low_mem()
        con.execute(
            """
            COPY (SELECT * FROM read_parquet($1) ORDER BY maid, timestamp)
            TO $2 (FORMAT PARQUET, COMPRESSION ZSTD)
            """,
            [out_path, tmp_sorted],
        )
        con.close()
        os.replace(tmp_sorted, out_path)

    pq_size = os.path.getsize(out_path) if os.path.exists(out_path) else 0
    total_csv_bytes += month_csv_bytes
    total_pq_bytes += pq_size

    elapsed = time.time() - t0
    ratio = month_csv_bytes / pq_size if pq_size > 0 else 0
    print(f" -> {total_rows:,} baris | CSV: {month_csv_bytes / 1e6:.0f} MB -> Parquet: {pq_size / 1e6:.0f} MB ({ratio:.1f}x kompresi) | {elapsed:.1f}s")

KONVERSI CSV -> PARQUET

 [2021_10_Oktober] Mengkonversi 1 file CSV...
 [2021_10_Oktober] Dropped 24 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> 12,123,709 baris | CSV: 932 MB -> Parquet: 47 MB (19.8x kompresi) | 37.4s

 [2021_11_November] Mengkonversi 7 file CSV...
 [2021_11_November] Dropped 35 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> 119,072,431 baris | CSV: 8943 MB -> Parquet: 355 MB (25.2x kompresi) | 375.3s

 [2021_12_Desember] Mengkonversi 3 file CSV...
 [2021_12_Desember] Dropped 32 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...


...

 -> 26,837,642 baris | CSV: 2004 MB -> Parquet: 90 MB (22.3x kompresi) | 72.6s

 [2022_04_April] Mengkonversi 1 file CSV...
 [2022_04_April] Dropped 4 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...
 -> 1,048,571 baris | CSV: 80 MB -> Parquet: 10 MB (8.2x kompresi) | 3.4s

 [2022_05_Mei] Mengkonversi 2 file CSV...
 [2022_05_Mei] Dropped 11 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> 31,075,618 baris | CSV: 2263 MB -> Parquet: 214 MB (10.6x kompresi) | 88.3s

 [2022_06_Juni] Mengkonversi 1 file CSV...
 [2022_06_Juni] Dropped 27 baris dengan NaN (latitude/longitude/timestamp)
 Sorting berdasarkan maid, timestamp...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 -> 12,246,261 baris | CSV: 920 MB -> Parquet: 74 MB (12.5x kompresi) | 34.3s


In [7]:
# ==========================================
# 2. KONVERSI PeopleGraph
# ==========================================
PG_SRC = os.path.join(GPS_SRC, "PeopleGraph", "people_graph.csv")
pg_out = os.path.join(GPS_DST, "people_graph.parquet")

if os.path.exists(pg_out):
    print("\n [SKIP] people_graph.parquet sudah ada")
    total_pq_bytes += os.path.getsize(pg_out)
    total_csv_bytes += os.path.getsize(PG_SRC)
else:
    print("\n [PeopleGraph] Mengkonversi...", flush=True)
    t0 = time.time()
    pg_csv_size = os.path.getsize(PG_SRC)

    pg_cols = [
        "maid", "gender", "col3", "col4", "country", "geohash", "income",
        "province", "kabupaten", "kecamatan", "kelurahan",
    ]

    # PeopleGraph tidak punya header
    df_pg = pd.read_csv(
        PG_SRC, header=None, names=pg_cols, quotechar='"',
        na_values=[r"\N"], dtype=str
    )

    # Hapus kolom kosong / tidak berguna
    df_pg = df_pg.drop(columns=["col3", "col4"], errors="ignore")

    # Hapus baris duplikat
    before = len(df_pg)
    dropped_pg = df_pg[df_pg.duplicated()]
    df_pg = df_pg.drop_duplicates()
    after = len(df_pg)
    if before > after:
        print(f" Hapus {before - after:,} baris duplikat ({before:,} -> {after:,})")
        # Mirror path: convert-to-parquet/DataGPS/PeopleGraph/people_graph_dropped.csv
        rel_path = os.path.relpath(PG_SRC, BASE_DIR)
        stem, ext = os.path.splitext(rel_path)
        dropped_pg_path = os.path.join(DROPPED_DIR, stem + "_dropped" + ext)
        os.makedirs(os.path.dirname(dropped_pg_path), exist_ok=True)
        dropped_pg.to_csv(dropped_pg_path, index=False)

    # Konversi ke kategorikal (hemat memori untuk kolom low-cardinality)
    for col in [
        "gender", "country", "income", "province", "kabupaten", "kecamatan", "kelurahan",
    ]:
        if col in df_pg.columns:
            df_pg[col] = df_pg[col].astype("category")

    # Simpan ke Parquet (tanpa sort dulu)
    table = pa.Table.from_pandas(df_pg, preserve_index=False)
    n_rows_pg = len(df_pg)
    del df_pg  # bebaskan RAM sebelum sort
    pq.write_table(
        table,
        pg_out,
        compression="zstd",
        use_dictionary=[
            "gender", "country", "income", "province", "kabupaten", "kecamatan", "kelurahan",
        ],
    )
    del table

    # Sorting berdasarkan maid (DuckDB out-of-core, hemat RAM)
    print(" Sorting berdasarkan maid...", flush=True)
    pg_tmp = pg_out + ".sorting.tmp"
    con = duckdb_low_mem()
    con.execute(
        """
        COPY (SELECT * FROM read_parquet($1) ORDER BY maid)
        TO $2 (FORMAT PARQUET, COMPRESSION ZSTD)
        """,
        [pg_out, pg_tmp],
    )
    con.close()
    os.replace(pg_tmp, pg_out)

    pg_pq_size = os.path.getsize(pg_out)
    total_csv_bytes += pg_csv_size
    total_pq_bytes += pg_pq_size

    elapsed = time.time() - t0
    ratio = pg_csv_size / pg_pq_size if pg_pq_size > 0 else 0
    print(f" -> {n_rows_pg:,} baris | CSV: {pg_csv_size / 1e6:.0f} MB -> Parquet: {pg_pq_size / 1e6:.0f} MB ({ratio:.1f}x kompresi) | {elapsed:.1f}s")


 [PeopleGraph] Mengkonversi...
 Hapus 589,064 baris duplikat (1,834,289 -> 1,245,225)
 Sorting berdasarkan maid...
 -> 1,245,225 baris | CSV: 251 MB -> Parquet: 31 MB (8.1x kompresi) | 9.5s


In [8]:
# ==========================================
# 3. KONVERSI DataMPD
# ==========================================
MPD_SRC = os.path.join(BASE_DIR, "DataMPD", "mpd_sample_small.csv")
MPD_DST = os.path.join(BASE_DIR, "DataMPD_parquet")
os.makedirs(MPD_DST, exist_ok=True)
mpd_out = os.path.join(MPD_DST, "mpd_sample_small.parquet")

if os.path.exists(mpd_out):
    print("\n [SKIP] mpd_sample_small.parquet sudah ada")
    total_pq_bytes += os.path.getsize(mpd_out)
    total_csv_bytes += os.path.getsize(MPD_SRC)
else:
    print("\n [DataMPD] Mengkonversi...", flush=True)
    t0 = time.time()
    mpd_csv_size = os.path.getsize(MPD_SRC)

    df_mpd = pd.read_csv(MPD_SRC, na_values=[r"\N"])

    table = pa.Table.from_pandas(df_mpd, preserve_index=False)
    pq.write_table(table, mpd_out, compression="zstd")

    mpd_pq_size = os.path.getsize(mpd_out)
    total_csv_bytes += mpd_csv_size
    total_pq_bytes += mpd_pq_size

    elapsed = time.time() - t0
    ratio = mpd_csv_size / mpd_pq_size if mpd_pq_size > 0 else 0
    print(f" -> {len(df_mpd):,} baris | CSV: {mpd_csv_size / 1e6:.1f} MB -> Parquet: {mpd_pq_size / 1e6:.1f} MB ({ratio:.1f}x kompresi) | {elapsed:.1f}s")


 [DataMPD] Mengkonversi...
 -> 1,113 baris | CSV: 0.1 MB -> Parquet: 0.0 MB (1.7x kompresi) | 0.0s


In [9]:
# ==========================================
# RINGKASAN
# ==========================================
print("\n" + "=" * 60)
print("RINGKASAN KONVERSI")
print("=" * 60)
print(f" Total CSV    : {total_csv_bytes / 1e9:.2f} GB")
print(f" Total Parquet: {total_pq_bytes / 1e9:.2f} GB")
if total_pq_bytes > 0:
    print(f" Rasio kompresi: {total_csv_bytes / total_pq_bytes:.1f}x lebih kecil")
    print(f" Hemat         : {(total_csv_bytes - total_pq_bytes) / 1e9:.2f} GB ({(1 - total_pq_bytes / total_csv_bytes) * 100:.0f}%)")
print("=" * 60)
print("\nFile Parquet tersedia di:")
print(f" - {GPS_DST}/")
print(f" - {MPD_DST}/")
print("\nSelesai!")


RINGKASAN KONVERSI
 Total CSV    : 23.45 GB
 Total Parquet: 1.41 GB
 Rasio kompresi: 16.7x lebih kecil
 Hemat         : 22.04 GB (94%)

File Parquet tersedia di:
 - /home/repal/Github/skripsi/DataGPS_parquet/
 - /home/repal/Github/skripsi/DataMPD_parquet/

Selesai!
